# 03_supplementary_experiments_clean.ipynb

**Purpose.** Reproduce the supplementary and robustness experiments for the hurricane-resilient supply-chain study after Notebooks 01 and 02 have been run successfully.

This notebook is intentionally separated from the main ML and optimization notebooks. It contains reviewer-response / robustness analyses: temporal holdout, exposure-threshold diagnostics, ML probability vs. rule-based diagnostics, scenario-reduction diagnostics, and unmet-demand penalty sensitivity. It uses the finalized without-SEASON ML outputs and the corrected ML Dynamic optimization inputs from 01/02.


In [ ]:
# Reusable implementation imports
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from hurricane_resilience.data import load_event_node_data, read_csv, haversine_km
from hurricane_resilience.evaluation import weighted_cvar, extract_nondominated
from hurricane_resilience.paths import FIGURES_DIR, TABLES_DIR
from hurricane_resilience.scenarios import validate_scenario_weights, kmeans_reduce


In [ ]:
# Cell 00: Global setup, paths, reproducibility, and safe file utilities

import os
import sys
import math
import json
import warnings
from pathlib import Path

# Windows + non-ASCII path safety for sklearn/joblib multiprocessing.
# Keep all sklearn/joblib operations single-threaded to avoid loky ASCII path issues.
import tempfile
JOBLIB_TEMP = Path(tempfile.gettempdir()) / "hurricane_resilience_joblib"
try:
    JOBLIB_TEMP.mkdir(parents=True, exist_ok=True)
    os.environ["JOBLIB_TEMP_FOLDER"] = str(JOBLIB_TEMP)
    os.environ["TMP"] = str(JOBLIB_TEMP)
    os.environ["TEMP"] = str(JOBLIB_TEMP)
except Exception:
    # Non-Windows environments may not allow creating C:/joblib_temp. Single-threading below is still sufficient.
    pass

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = None

RANDOM_STATE = 42
SKLEARN_N_JOBS = 1
NOTEBOOK_VERSION = "03_clean_v2_after_01v7_02v3"

# Run this notebook from the same folder that contains the outputs from 01 and 02.
BASE_DIR = PROJECT_ROOT

# If your notebook is opened from another folder, manually set BASE_DIR here, for example:
# PROJECT_ROOT is resolved above from this notebook location

OUTPUT_DIR = BASE_DIR / "outputs_03_supplementary"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_DPI = 300

# For time-consuming supplementary optimization experiments, load existing outputs when available.
# If the corresponding files are missing and Gurobi is available, the notebook recomputes them.
REUSE_EXISTING_RESULTS = True

SEARCH_DIRS = [
    BASE_DIR,
    BASE_DIR / "outputs_01_ml_pipeline",
    BASE_DIR / "outputs_02_optimization",
    BASE_DIR / "outputs_03_supplementary",
    OUTPUT_DIR,
]

print("Notebook version:", NOTEBOOK_VERSION)
print("Working directory:", BASE_DIR)
print("Output directory:", OUTPUT_DIR)
print("SKLEARN_N_JOBS:", SKLEARN_N_JOBS)
print("REUSE_EXISTING_RESULTS:", REUSE_EXISTING_RESULTS)


def find_file(filename, patterns=None, required=True):
    """Find a file in the project folder or output folders, with optional fallback glob patterns."""
    patterns = patterns or []
    for directory in SEARCH_DIRS:
        exact = directory / filename
        if exact.exists():
            return exact

    candidates = []
    for directory in SEARCH_DIRS:
        for pattern in patterns:
            candidates.extend(directory.glob(pattern))
    candidates = sorted(set(candidates))

    if candidates:
        print(f"Using fallback for {filename}: {candidates[0]}")
        return candidates[0]

    if required:
        available = sorted([p.name for p in BASE_DIR.glob("*.csv")])
        preview = "\n".join(available[:150])
        searched = "\n".join(str(d) for d in SEARCH_DIRS)
        raise FileNotFoundError(
            f"Cannot find required file: {filename}\n"
            f"Searched folders:\n{searched}\n"
            f"Fallback patterns: {patterns}\n"
            f"Available CSV files in BASE_DIR include:\n{preview}"
        )
    return None


def save_csv(df, filename, output_dir=OUTPUT_DIR, also_root=True):
    """Save a CSV into outputs_03 and optionally to project root with the manuscript filename."""
    path = output_dir / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    if also_root:
        df.to_csv(BASE_DIR / filename, index=False, encoding="utf-8-sig")
    print("Saved:", path)
    return path


def save_figure(filename, output_dir=OUTPUT_DIR, dpi=FIGURE_DPI):
    path = output_dir / filename
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print("Saved:", path)
    return path


def maybe_load_csv(filename, patterns=None):
    if not REUSE_EXISTING_RESULTS:
        return None
    path = find_file(filename, patterns=patterns, required=False)
    if path is not None:
        print("Loaded existing:", path)
        return pd.read_csv(path)
    return None


def display_if_available(obj, max_rows=20):
    if isinstance(obj, pd.DataFrame):
        if display is not None:
            display(obj.head(max_rows))
        else:
            print(obj.head(max_rows).to_string())
    else:
        if display is not None:
            display(obj)
        else:
            print(obj)


In [ ]:
# Cell 01: Load finalized 01/02 inputs and build missing optimization network files if needed

storm_node_path = find_file(
    "storm_node_dataset_14nodes_200km_50kt.csv",
    patterns=["*storm*node*14nodes*200km*50kt*.csv", "*storm*node*14nodes*.csv"],
)

# Prefer 01 v7's full probability matrix. If it is missing, reconstruct it from final OOF predictions.
full_prob_path = find_file(
    "final_scenario_node_probabilities_full_without_season.csv",
    patterns=["*scenario_node_probabilities*full*without_season*.csv"],
    required=False,
)
risk_pred_path = find_file(
    "final_risk_predictions_oof_without_season.csv",
    patterns=["*risk_predictions*oof*without_season*.csv"],
    required=False,
)
hybrid_scenario_path = find_file(
    "final_hybrid_reduced_hurricane_scenarios_without_season.csv",
    patterns=["*hybrid*reduced*hurricane*without_season*.csv"],
)
risk_matrix_path = find_file(
    "final_optimization_scenario_node_risk_matrix_without_season.csv",
    patterns=["*optimization*scenario*node*risk*without_season*.csv"],
)

NODE_PARAM_FILE = "optimization_node_parameters.csv"
ARC_FILE = "optimization_transport_arcs.csv"
node_params_path = find_file(NODE_PARAM_FILE, patterns=["*node*parameters*.csv"], required=False)
arcs_path = find_file(ARC_FILE, patterns=["*transport*arcs*.csv"], required=False)


def haversine_km_scalar(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2.0) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2.0) ** 2
    return 2.0 * R * math.asin(math.sqrt(a))


def create_node_parameters():
    """Create the same stylized 14-node supply-chain network used by Notebook 02."""
    return pd.DataFrame([
        {"node_id": "N1",  "node_name": "Houston",        "role": "source", "fixed_cost": 0,    "capacity": 900,  "supply": 900,  "demand": 0,   "latitude": 29.7604, "longitude": -95.3698},
        {"node_id": "N2",  "node_name": "New Orleans",    "role": "market", "fixed_cost": 0,    "capacity": 0,    "supply": 0,    "demand": 420, "latitude": 29.9511, "longitude": -90.0715},
        {"node_id": "N3",  "node_name": "Miami",          "role": "market", "fixed_cost": 0,    "capacity": 0,    "supply": 0,    "demand": 520, "latitude": 25.7617, "longitude": -80.1918},
        {"node_id": "N4",  "node_name": "Orlando",        "role": "dc",     "fixed_cost": 1800, "capacity": 780,  "supply": 0,    "demand": 0,   "latitude": 28.5383, "longitude": -81.3792},
        {"node_id": "N5",  "node_name": "Atlanta",        "role": "dc",     "fixed_cost": 2100, "capacity": 1000, "supply": 0,    "demand": 0,   "latitude": 33.7490, "longitude": -84.3880},
        {"node_id": "N6",  "node_name": "Dallas",         "role": "source", "fixed_cost": 0,    "capacity": 1100, "supply": 1100, "demand": 0,   "latitude": 32.7767, "longitude": -96.7970},
        {"node_id": "N7",  "node_name": "Tampa",          "role": "market", "fixed_cost": 0,    "capacity": 0,    "supply": 0,    "demand": 380, "latitude": 27.9506, "longitude": -82.4572},
        {"node_id": "N8",  "node_name": "Jacksonville",   "role": "dc",     "fixed_cost": 1600, "capacity": 700,  "supply": 0,    "demand": 0,   "latitude": 30.3322, "longitude": -81.6557},
        {"node_id": "N9",  "node_name": "Mobile",         "role": "source", "fixed_cost": 0,    "capacity": 650,  "supply": 650,  "demand": 0,   "latitude": 30.6954, "longitude": -88.0399},
        {"node_id": "N10", "node_name": "Savannah",       "role": "market", "fixed_cost": 0,    "capacity": 0,    "supply": 0,    "demand": 300, "latitude": 32.0809, "longitude": -81.0912},
        {"node_id": "N11", "node_name": "Charleston",     "role": "market", "fixed_cost": 0,    "capacity": 0,    "supply": 0,    "demand": 280, "latitude": 32.7765, "longitude": -79.9311},
        {"node_id": "N12", "node_name": "Corpus Christi", "role": "source", "fixed_cost": 0,    "capacity": 700,  "supply": 700,  "demand": 0,   "latitude": 27.8006, "longitude": -97.3964},
        {"node_id": "N13", "node_name": "Baton Rouge",    "role": "dc",     "fixed_cost": 1300, "capacity": 620,  "supply": 0,    "demand": 0,   "latitude": 30.4515, "longitude": -91.1871},
        {"node_id": "N14", "node_name": "Charlotte",      "role": "dc",     "fixed_cost": 1700, "capacity": 760,  "supply": 0,    "demand": 0,   "latitude": 35.2271, "longitude": -80.8431},
    ])


def create_transport_arcs(node_params):
    sources_local = node_params[node_params["role"] == "source"]["node_id"].tolist()
    dcs_local = node_params[node_params["role"] == "dc"]["node_id"].tolist()
    markets_local = node_params[node_params["role"] == "market"]["node_id"].tolist()
    loc = node_params.set_index("node_id")[["latitude", "longitude"]].to_dict("index")
    rows = []
    cost_per_km = 0.015

    for i in sources_local:
        for j in dcs_local:
            dist = haversine_km_scalar(loc[i]["latitude"], loc[i]["longitude"], loc[j]["latitude"], loc[j]["longitude"])
            rows.append({"from_node": i, "to_node": j, "arc_type": "source_dc", "distance_km": dist, "unit_transport_cost": dist * cost_per_km})

    for j in dcs_local:
        for m in markets_local:
            dist = haversine_km_scalar(loc[j]["latitude"], loc[j]["longitude"], loc[m]["latitude"], loc[m]["longitude"])
            rows.append({"from_node": j, "to_node": m, "arc_type": "dc_market", "distance_km": dist, "unit_transport_cost": dist * cost_per_km})

    return pd.DataFrame(rows)


df = pd.read_csv(storm_node_path)
df["disrupted"] = df["disrupted"].astype(int)

if full_prob_path is not None:
    prob_df = pd.read_csv(full_prob_path)
elif risk_pred_path is not None:
    print("Full probability matrix missing; reconstructing from final OOF predictions.")
    pred = pd.read_csv(risk_pred_path)
    required_pred_cols = {"SID", "SEASON", "NAME", "node_id", "node_name", "node_type", "p_uncalibrated", "p_calibrated", "disrupted"}
    missing_pred_cols = required_pred_cols - set(pred.columns)
    if missing_pred_cols:
        raise ValueError(f"Cannot reconstruct full probability matrix; prediction file missing columns: {sorted(missing_pred_cols)}")
    prob_df = pred[["SID", "SEASON", "NAME", "node_id", "node_name", "node_type", "p_uncalibrated", "p_calibrated", "disrupted"]].copy()
    prob_df = prob_df.rename(columns={"SID": "scenario_id", "p_uncalibrated": "p_is_uncalibrated", "p_calibrated": "p_is_calibrated"})
    prob_df["scenario_weight"] = 1.0 / prob_df["scenario_id"].nunique()
    save_csv(prob_df, "final_scenario_node_probabilities_full_without_season.csv", output_dir=BASE_DIR, also_root=False)
else:
    raise FileNotFoundError("Need either final_scenario_node_probabilities_full_without_season.csv or final_risk_predictions_oof_without_season.csv.")

hybrid_scenarios = pd.read_csv(hybrid_scenario_path)
risk_matrix = pd.read_csv(risk_matrix_path)

if node_params_path is None:
    print("Node-parameter file missing; creating optimization_node_parameters.csv from embedded study network.")
    node_params = create_node_parameters()
    save_csv(node_params, NODE_PARAM_FILE, output_dir=BASE_DIR, also_root=False)
else:
    node_params = pd.read_csv(node_params_path)

if arcs_path is None:
    print("Transport-arc file missing; creating optimization_transport_arcs.csv from embedded coordinates.")
    arcs_df = create_transport_arcs(node_params)
    save_csv(arcs_df, ARC_FILE, output_dir=BASE_DIR, also_root=False)
else:
    arcs_df = pd.read_csv(arcs_path)

# Structural validation. These assertions protect the supplementary notebook from silently using wrong inputs.
required_df_cols = {
    "SID", "SEASON", "NAME", "node_id", "node_name", "node_type", "month",
    "duration_hours", "max_wind_kts", "mean_wind_kts", "mean_storm_speed",
    "max_storm_speed", "min_distance_km", "coastal", "disrupted"
}
required_prob_cols = {"scenario_id", "SEASON", "NAME", "node_id", "node_name", "node_type", "p_is_uncalibrated", "p_is_calibrated", "disrupted", "scenario_weight"}
required_risk_cols = {"scenario_id", "node_id", "p_is", "p_is_sensitivity", "disrupted", "scenario_weight"}
required_node_cols = {"node_id", "node_name", "role", "fixed_cost", "capacity", "supply", "demand", "latitude", "longitude"}
required_arc_cols = {"from_node", "to_node", "arc_type", "distance_km", "unit_transport_cost"}

for name, data, cols in [
    ("storm-node dataset", df, required_df_cols),
    ("full probability matrix", prob_df, required_prob_cols),
    ("optimization risk matrix", risk_matrix, required_risk_cols),
    ("node parameters", node_params, required_node_cols),
    ("transport arcs", arcs_df, required_arc_cols),
]:
    missing = cols - set(data.columns)
    if missing:
        raise ValueError(f"{name} missing required columns: {sorted(missing)}")

if df.shape[0] != 6496 or df["SID"].nunique() != 464 or df["node_id"].nunique() != 14 or int(df["disrupted"].sum()) != 160:
    raise ValueError("Storm-node dataset does not match the finalized 01 v7 dimensions: 6,496 rows, 464 storms, 14 nodes, 160 positives.")

if risk_matrix["scenario_id"].nunique() != 29 or risk_matrix.shape[0] != 406:
    raise ValueError("Optimization risk matrix must contain 29 reduced scenarios and 406 rows = 29 × 14.")

nodes_per_scenario = risk_matrix.groupby("scenario_id")["node_id"].nunique()
if not (nodes_per_scenario == 14).all():
    raise ValueError("Every reduced scenario must contain exactly 14 nodes.")

weight_sum = risk_matrix[["scenario_id", "scenario_weight"]].drop_duplicates()["scenario_weight"].sum()
if not np.isclose(weight_sum, 1.0, atol=1e-6):
    raise ValueError(f"Scenario weights must sum to 1. Current sum: {weight_sum}")

print("Storm-node dataset:", df.shape)
print("Full probability matrix:", prob_df.shape)
print("Hybrid scenarios:", hybrid_scenarios.shape)
print("Optimization risk matrix:", risk_matrix.shape)
print("Node parameters:", node_params.shape)
print("Transport arcs:", arcs_df.shape)
print("Input validation passed.")


In [ ]:
# Cell 02: Define reusable ML preprocessing, metrics, and model-pool utilities
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    f1_score,
    recall_score,
    precision_score,
)
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV

final_numeric_features = [
    "month",
    "duration_hours",
    "max_wind_kts",
    "mean_wind_kts",
    "mean_storm_speed",
    "max_storm_speed",
    "min_distance_km",
    "coastal",
]
final_categorical_features = ["node_type"]
final_features = final_numeric_features + final_categorical_features

with_season_numeric_features = ["SEASON"] + final_numeric_features
with_season_features = with_season_numeric_features + final_categorical_features

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop",
    )

def make_model_pipeline(model, numeric_features=final_numeric_features, categorical_features=final_categorical_features):
    return Pipeline(
        steps=[
            ("preprocess", make_preprocessor(numeric_features, categorical_features)),
            ("model", model),
        ]
    )

def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for b in range(n_bins):
        left, right = bins[b], bins[b + 1]
        if b == n_bins - 1:
            mask = (y_prob >= left) & (y_prob <= right)
        else:
            mask = (y_prob >= left) & (y_prob < right)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(y_true)) * abs(y_true[mask].mean() - y_prob[mask].mean())
    return float(ece)

def evaluate_binary_probabilities(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_prob_clipped = np.clip(y_prob, 1e-8, 1 - 1e-8)
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "brier_score": brier_score_loss(y_true, y_prob_clipped),
        "ece": expected_calibration_error(y_true, y_prob_clipped, n_bins=10),
        "f1_at_05": f1_score(y_true, y_pred, zero_division=0),
        "recall_at_05": recall_score(y_true, y_pred, zero_division=0),
        "precision_at_05": precision_score(y_true, y_pred, zero_division=0),
    }
    if len(np.unique(y_true)) == 2:
        metrics.update({
            "roc_auc": roc_auc_score(y_true, y_prob_clipped),
            "average_precision": average_precision_score(y_true, y_prob_clipped),
            "log_loss": log_loss(y_true, y_prob_clipped, labels=[0, 1]),
        })
    else:
        metrics.update({"roc_auc": np.nan, "average_precision": np.nan, "log_loss": np.nan})
    return metrics

def get_random_forest():
    return RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=3,
        class_weight="balanced",
        n_jobs=SKLEARN_N_JOBS,
        random_state=RANDOM_STATE,
    )

def get_candidate_models():
    models = {
        "Logistic Regression": LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE),
        "SVM": SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE),
        "Random Forest": get_random_forest(),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=1000,
            early_stopping=True,
            random_state=RANDOM_STATE,
        ),
    }
    try:
        from xgboost import XGBClassifier
        models["XGBoost"] = XGBClassifier(
            n_estimators=400,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=SKLEARN_N_JOBS,
            random_state=RANDOM_STATE,
        )
    except Exception as e:
        print("XGBoost is not available and will be skipped:", e)
    try:
        from lightgbm import LGBMClassifier
        models["LightGBM"] = LGBMClassifier(
            n_estimators=400,
            learning_rate=0.03,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            verbose=-1,
        )
    except Exception as e:
        print("LightGBM is not available and will be skipped:", e)
    order = ["Logistic Regression", "SVM", "Random Forest", "Gradient Boosting", "XGBoost", "LightGBM", "MLP"]
    return {name: models[name] for name in order if name in models}

def make_group_cv(y, groups, max_splits=5):
    y = np.asarray(y).astype(int)
    groups = np.asarray(groups)
    positive_groups = pd.Series(groups[y == 1]).nunique()
    negative_groups = pd.Series(groups[y == 0]).nunique()
    n_splits = int(min(max_splits, positive_groups, negative_groups))
    if n_splits < 2:
        raise ValueError(f"Not enough positive/negative groups for CV. positive_groups={positive_groups}, negative_groups={negative_groups}")
    return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)


In [ ]:
# Cell 03: Temporal holdout experiment, train 2000-2020 and test 2021-2025

existing_temporal = maybe_load_csv(
    "supp_temporal_holdout_model_results_2000_2020_to_2021_2025.csv",
    patterns=["*temporal*holdout*model*results*.csv"],
)

if existing_temporal is not None:
    temporal_results = existing_temporal.copy()
    print("Using existing temporal holdout results.")
else:
    train_end_year = 2020
    test_start_year = 2021

    train_df = df[df["SEASON"] <= train_end_year].copy()
    test_df = df[df["SEASON"] >= test_start_year].copy()

    X_train = train_df[final_features]
    y_train = train_df["disrupted"].astype(int)
    X_test = test_df[final_features]
    y_test = test_df["disrupted"].astype(int)

    if y_train.nunique() < 2 or y_test.nunique() < 2:
        raise ValueError("Temporal holdout requires both positive and negative samples in train and test periods.")

    print("Train years:", int(train_df["SEASON"].min()), "-", int(train_df["SEASON"].max()))
    print("Test years:", int(test_df["SEASON"].min()), "-", int(test_df["SEASON"].max()))
    print("Train positives:", int(y_train.sum()), "Test positives:", int(y_test.sum()))

    rows = []
    for model_name, model in get_candidate_models().items():
        print("Temporal holdout:", model_name)
        pipe = make_model_pipeline(model, final_numeric_features, final_categorical_features)
        pipe.fit(X_train, y_train)
        y_prob = pipe.predict_proba(X_test)[:, 1]
        metrics = evaluate_binary_probabilities(y_test, y_prob)
        row = {
            "model": model_name,
            "train_years": f"{train_df['SEASON'].min()}-{train_df['SEASON'].max()}",
            "test_years": f"{test_df['SEASON'].min()}-{test_df['SEASON'].max()}",
            "train_samples": len(train_df),
            "test_samples": len(test_df),
            "train_positive": int(y_train.sum()),
            "test_positive": int(y_test.sum()),
            "train_positive_ratio": float(y_train.mean()),
            "test_positive_ratio": float(y_test.mean()),
        }
        row.update(metrics)
        rows.append(row)

    temporal_results = pd.DataFrame(rows).sort_values(by=["average_precision", "roc_auc"], ascending=False).reset_index(drop=True)
    save_csv(temporal_results, "supp_temporal_holdout_model_results_2000_2020_to_2021_2025.csv", also_root=True)

required_temporal_cols = {"model", "roc_auc", "average_precision", "brier_score", "f1_at_05"}
missing_temporal_cols = required_temporal_cols - set(temporal_results.columns)
if missing_temporal_cols:
    raise ValueError(f"Temporal holdout result missing required columns: {sorted(missing_temporal_cols)}")

display_if_available(temporal_results)


In [ ]:
# Cell 04: Exposure-threshold positive-ratio and RF group-CV sensitivity

threshold_file = find_file("threshold_summary_14nodes.csv", patterns=["*threshold*summary*.csv"], required=False)
if threshold_file is not None:
    threshold_summary = pd.read_csv(threshold_file)
else:
    threshold_summary = pd.DataFrame()

if not threshold_summary.empty:
    threshold_summary = threshold_summary.copy()
    threshold_summary["positive_ratio_percent"] = threshold_summary["positive_ratio"] * 100
    threshold_summary["is_main_threshold"] = (
        (threshold_summary["distance_threshold_km"] == 200) & (threshold_summary["wind_threshold_kts"] == 50)
    )
    save_csv(threshold_summary, "supp_threshold_positive_ratio_summary.csv", also_root=True)
    display_if_available(threshold_summary, max_rows=30)
else:
    print("threshold_summary_14nodes.csv is not available; positive-ratio summary is skipped.")

existing_threshold_rf = maybe_load_csv(
    "supp_threshold_rf_groupcv_available_thresholds.csv",
    patterns=["*threshold*rf*groupcv*available*.csv"],
)

if existing_threshold_rf is not None:
    threshold_rf_results = existing_threshold_rf.copy()
    print("Using existing threshold RF group-CV results.")
else:
    available_settings = []
    if "max_wind_within_100km" in df.columns:
        available_settings.extend([
            (100, 34, "max_wind_within_100km"),
            (100, 50, "max_wind_within_100km"),
            (100, 64, "max_wind_within_100km"),
        ])
    if "max_wind_within_200km" in df.columns:
        available_settings.extend([
            (200, 34, "max_wind_within_200km"),
            (200, 50, "max_wind_within_200km"),
            (200, 64, "max_wind_within_200km"),
        ])

    if not available_settings:
        raise ValueError("No max_wind_within_*km columns are available for threshold RF sensitivity.")

    def run_rf_group_cv_for_target(temp_df, target_col, n_splits=5):
        X = temp_df[final_features].copy()
        y = temp_df[target_col].astype(int).values
        groups = temp_df["SID"].values
        try:
            cv = make_group_cv(y, groups, max_splits=n_splits)
        except ValueError:
            return {
                "n_splits": 0,
                "roc_auc_mean": np.nan,
                "average_precision_mean": np.nan,
                "brier_score_mean": np.nan,
                "log_loss_mean": np.nan,
                "ece_mean": np.nan,
                "f1_at_05_mean": np.nan,
            }
        rows = []
        for fold_id, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
            pipe = make_model_pipeline(get_random_forest(), final_numeric_features, final_categorical_features)
            pipe.fit(X.iloc[train_idx], y[train_idx])
            prob = pipe.predict_proba(X.iloc[test_idx])[:, 1]
            m = evaluate_binary_probabilities(y[test_idx], prob)
            m["fold"] = fold_id
            rows.append(m)
        fold_df = pd.DataFrame(rows)
        out = {"n_splits": fold_df["fold"].nunique()}
        for col in ["roc_auc", "average_precision", "brier_score", "log_loss", "ece", "f1_at_05", "recall_at_05", "precision_at_05"]:
            out[f"{col}_mean"] = fold_df[col].mean()
            out[f"{col}_std"] = fold_df[col].std()
        return out

    threshold_rows = []
    for distance_km, wind_kts, wind_col in available_settings:
        label_col = f"label_{distance_km}km_{wind_kts}kt"
        temp_df = df.copy()
        temp_df[label_col] = (temp_df[wind_col] >= wind_kts).astype(int)
        perf = run_rf_group_cv_for_target(temp_df, label_col, n_splits=5)
        row = {
            "distance_threshold_km": distance_km,
            "wind_threshold_kts": wind_kts,
            "wind_column_used": wind_col,
            "positive_count": int(temp_df[label_col].sum()),
            "total_count": len(temp_df),
            "positive_ratio": float(temp_df[label_col].mean()),
            "is_main_threshold": (distance_km == 200 and wind_kts == 50),
        }
        row.update(perf)
        threshold_rows.append(row)

    threshold_rf_results = pd.DataFrame(threshold_rows).sort_values(by=["distance_threshold_km", "wind_threshold_kts"])
    save_csv(threshold_rf_results, "supp_threshold_rf_groupcv_available_thresholds.csv", also_root=True)

required_threshold_cols = {"distance_threshold_km", "wind_threshold_kts", "positive_count", "positive_ratio"}
missing_threshold_cols = required_threshold_cols - set(threshold_rf_results.columns)
if missing_threshold_cols:
    raise ValueError(f"Threshold RF sensitivity result missing columns: {sorted(missing_threshold_cols)}")

display_if_available(threshold_rf_results, max_rows=30)


In [ ]:
# Cell 05: ML probability vs rule-based binary exposure diagnostics
prob_col = "p_is_calibrated"
rule_col = "disrupted"

# Add physical explanatory columns for interpretability when available.
merge_cols = [
    "SID", "node_id", "min_distance_km", "max_wind_kts", "mean_wind_kts",
    "wind_at_closest", "max_wind_within_100km", "max_wind_within_200km"
]
existing_merge_cols = [c for c in merge_cols if c in df.columns]
phys = df[existing_merge_cols].rename(columns={"SID": "scenario_id"}).drop_duplicates(["scenario_id", "node_id"])
prob_aug = prob_df.merge(phys, on=["scenario_id", "node_id"], how="left")

rule0_high_ml = prob_aug[prob_aug[rule_col] == 0].sort_values(by=prob_col, ascending=False).head(20).copy()
rule0_high_ml["diagnostic_type"] = "Rule=0 but high ML probability"
rule1_low_ml = prob_aug[prob_aug[rule_col] == 1].sort_values(by=prob_col, ascending=True).head(20).copy()
rule1_low_ml["diagnostic_type"] = "Rule=1 but low ML probability"
examples = pd.concat([rule0_high_ml, rule1_low_ml], ignore_index=True)

front_cols = ["diagnostic_type", "scenario_id", "SEASON", "NAME", "node_id", "node_name", "node_type", "disrupted", "p_is_calibrated", "p_is_uncalibrated"]
extra_cols = [c for c in ["min_distance_km", "max_wind_kts", "mean_wind_kts", "wind_at_closest", "max_wind_within_100km", "max_wind_within_200km"] if c in examples.columns]
examples = examples[[c for c in front_cols + extra_cols if c in examples.columns]]

save_csv(examples, "supp_ml_probability_vs_rule_examples.csv", also_root=True)

distribution = (
    prob_aug.groupby(rule_col)[prob_col]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
    .reset_index()
)
save_csv(distribution, "supp_ml_probability_distribution_by_rule_label.csv", also_root=True)
display(distribution)
display(examples.head(10))


In [ ]:
# Cell 06: Scenario reduction diagnostics: Pure KMeans vs Pure Top-risk vs Hybrid final
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

scenario_agg = (
    prob_df.groupby(["scenario_id", "SEASON", "NAME"], as_index=False)
    .agg(
        total_predicted_risk=("p_is_calibrated", "sum"),
        mean_predicted_risk=("p_is_calibrated", "mean"),
        max_predicted_risk=("p_is_calibrated", "max"),
        observed_exposure_count=("disrupted", "sum"),
        scenario_weight=("scenario_weight", "first"),
    )
)
K = hybrid_scenarios["scenario_id"].nunique()
feature_cols = ["total_predicted_risk", "mean_predicted_risk", "max_predicted_risk", "observed_exposure_count"]
X_scen = np.log1p(scenario_agg[feature_cols].fillna(0.0))
X_scaled = StandardScaler().fit_transform(X_scen)

kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=50)
labels = kmeans.fit_predict(X_scaled)
scenario_agg["kmeans_cluster"] = labels
pure_kmeans_ids = []
for cluster_id in range(K):
    idx = np.where(labels == cluster_id)[0]
    centroid = kmeans.cluster_centers_[cluster_id]
    distances = np.linalg.norm(X_scaled[idx] - centroid, axis=1)
    pure_kmeans_ids.append(scenario_agg.iloc[idx[np.argmin(distances)]]["scenario_id"])
pure_kmeans_ids = set(pure_kmeans_ids)
pure_toprisk_ids = set(scenario_agg.sort_values(by="total_predicted_risk", ascending=False).head(K)["scenario_id"])
hybrid_ids = set(hybrid_scenarios["scenario_id"])

def summarize_selection(name, ids):
    ids = set(ids)
    selected = scenario_agg[scenario_agg["scenario_id"].isin(ids)].copy()
    total_weighted_risk_all = (scenario_agg["total_predicted_risk"] * scenario_agg["scenario_weight"]).sum()
    total_weighted_risk_selected = (selected["total_predicted_risk"] * selected["scenario_weight"]).sum()
    row = {
        "selection_method": name,
        "selected_count": len(selected),
        "scenario_weight_sum_original": selected["scenario_weight"].sum(),
        "mean_total_predicted_risk": selected["total_predicted_risk"].mean(),
        "max_total_predicted_risk": selected["total_predicted_risk"].max(),
        "mean_max_node_risk": selected["max_predicted_risk"].mean(),
        "max_node_risk": selected["max_predicted_risk"].max(),
        "mean_observed_exposure_count": selected["observed_exposure_count"].mean(),
        "max_observed_exposure_count": selected["observed_exposure_count"].max(),
        "weighted_total_risk_share_original_weights": total_weighted_risk_selected / total_weighted_risk_all,
    }
    for metric in ["total_predicted_risk", "max_predicted_risk", "observed_exposure_count"]:
        for top_k in [5, 10, 20, 29]:
            top_ids = set(scenario_agg.sort_values(by=metric, ascending=False).head(top_k)["scenario_id"])
            row[f"retention_top{top_k}_{metric}"] = len(ids & top_ids) / top_k
    return row

diag = pd.DataFrame([
    summarize_selection("Pure KMeans", pure_kmeans_ids),
    summarize_selection("Pure Top-risk", pure_toprisk_ids),
    summarize_selection("Hybrid final", hybrid_ids),
])
save_csv(diag, "supp_scenario_reduction_diagnostics.csv", also_root=True)
display(diag)

selected_rows = []
for method, ids in [("Pure KMeans", pure_kmeans_ids), ("Pure Top-risk", pure_toprisk_ids), ("Hybrid final", hybrid_ids)]:
    temp = scenario_agg[scenario_agg["scenario_id"].isin(ids)].copy()
    temp["selection_method"] = method
    selected_rows.append(temp)
selected_compare = pd.concat(selected_rows, ignore_index=True)
save_csv(selected_compare, "supp_selected_scenarios_by_reduction_method.csv", also_root=True)

plot_df = scenario_agg.copy()
plot_df["selected_by_kmeans"] = plot_df["scenario_id"].isin(pure_kmeans_ids)
plot_df["selected_by_toprisk"] = plot_df["scenario_id"].isin(pure_toprisk_ids)
plot_df["selected_by_hybrid"] = plot_df["scenario_id"].isin(hybrid_ids)

plt.figure(figsize=(8, 5.5))
plt.scatter(plot_df["total_predicted_risk"], plot_df["max_predicted_risk"], alpha=0.30, label="All scenarios")
plt.scatter(plot_df.loc[plot_df["selected_by_kmeans"], "total_predicted_risk"], plot_df.loc[plot_df["selected_by_kmeans"], "max_predicted_risk"], marker="o", s=55, label="Pure KMeans")
plt.scatter(plot_df.loc[plot_df["selected_by_toprisk"], "total_predicted_risk"], plot_df.loc[plot_df["selected_by_toprisk"], "max_predicted_risk"], marker="^", s=55, label="Pure Top-risk")
plt.scatter(plot_df.loc[plot_df["selected_by_hybrid"], "total_predicted_risk"], plot_df.loc[plot_df["selected_by_hybrid"], "max_predicted_risk"], marker="x", s=75, label="Hybrid final")
plt.xlabel("Total Predicted Network Risk")
plt.ylabel("Maximum Node-level Predicted Risk")
plt.title("Scenario Reduction Diagnostic")
plt.legend()
plt.tight_layout()
save_figure("supp_scenario_reduction_diagnostic_plot.png")
plt.show()


## Unmet-demand penalty sensitivity

The following cells duplicate the optimization helper definitions needed for self-contained reproduction of the unmet-demand penalty sensitivity experiment. They use the same model specification as Notebook 02.

In [ ]:
# Cell 07: Define design parsing, weighted CVaR, nondominance, and transport-arc utilities
def parse_design_string(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none", "—", "-"}:
        return []
    return [x.strip() for x in text.split(",") if x.strip()]

def design_to_string(nodes):
    nodes = sorted([str(x) for x in nodes if str(x).strip()])
    return ",".join(nodes)

def compute_weighted_cvar(loss_dict, weight_dict, alpha=0.90):
    scenarios_local = list(loss_dict.keys())
    losses = np.array([loss_dict[s] for s in scenarios_local], dtype=float)
    weights = np.array([weight_dict[s] for s in scenarios_local], dtype=float)
    weights = weights / weights.sum()
    candidate_etas = np.unique(np.concatenate(([0.0], losses)))
    best_cvar, best_eta = np.inf, None
    for eta in candidate_etas:
        cvar_value = eta + (1.0 / (1.0 - alpha)) * np.sum(weights * np.maximum(losses - eta, 0.0))
        if cvar_value < best_cvar:
            best_cvar, best_eta = cvar_value, eta
    return float(best_cvar), float(best_eta)

def extract_realized_solution_metrics(solution, scenarios, markets, scenario_weights, alpha=0.90):
    if not solution or "unmet_vars" not in solution:
        return {
            "realized_expected_unmet_demand": np.nan,
            "realized_cvar_unmet_demand": np.nan,
            "realized_eta_cvar": np.nan,
            "scenario_unmet": {},
        }
    unmet_vars = solution["unmet_vars"]
    scenario_unmet = {s: sum(unmet_vars[s, m].X for m in markets) for s in scenarios}
    expected_unmet = sum(scenario_weights[s] * scenario_unmet[s] for s in scenarios)
    cvar, eta = compute_weighted_cvar(scenario_unmet, scenario_weights, alpha=alpha)
    return {
        "realized_expected_unmet_demand": float(expected_unmet),
        "realized_cvar_unmet_demand": float(cvar),
        "realized_eta_cvar": float(eta),
        "scenario_unmet": scenario_unmet,
    }

def extract_nondominated(df, cost_col="total_cost", cvar_col="realized_cvar_unmet_demand", tol=1e-6):
    temp = df.dropna(subset=[cost_col, cvar_col]).copy().reset_index(drop=True)
    keep = []
    for i, row_i in temp.iterrows():
        dominated = False
        for j, row_j in temp.iterrows():
            if i == j:
                continue
            no_worse = (row_j[cost_col] <= row_i[cost_col] + tol) and (row_j[cvar_col] <= row_i[cvar_col] + tol)
            strictly_better = (row_j[cost_col] < row_i[cost_col] - tol) or (row_j[cvar_col] < row_i[cvar_col] - tol)
            if no_worse and strictly_better:
                dominated = True
                break
        if not dominated:
            keep.append(i)
    return temp.loc[keep].sort_values(by=[cvar_col, cost_col]).reset_index(drop=True)

def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return float(2 * radius * np.arcsin(np.sqrt(a)))


In [ ]:
# Cell 08: Prepare sets, dictionaries, and main optimization parameters
sources = node_params[node_params["role"] == "source"]["node_id"].tolist()
dcs = node_params[node_params["role"] == "dc"]["node_id"].tolist()
markets = node_params[node_params["role"] == "market"]["node_id"].tolist()
scenarios = risk_matrix["scenario_id"].unique().tolist()

scenario_weights = (
    risk_matrix[["scenario_id", "scenario_weight"]]
    .drop_duplicates()
    .set_index("scenario_id")["scenario_weight"]
    .to_dict()
)
# Normalize defensively in case input weights are not exactly normalized.
weight_sum = sum(scenario_weights.values())
scenario_weights = {s: w / weight_sum for s, w in scenario_weights.items()}

p_is = risk_matrix.set_index(["scenario_id", "node_id"])["p_is"].to_dict()
p_is_sensitivity = (
    risk_matrix.set_index(["scenario_id", "node_id"])["p_is_sensitivity"].to_dict()
    if "p_is_sensitivity" in risk_matrix.columns else p_is.copy()
)
rule_binary = risk_matrix.set_index(["scenario_id", "node_id"])["disrupted"].astype(float).to_dict()

fixed_cost = node_params.set_index("node_id")["fixed_cost"].to_dict()
capacity = node_params.set_index("node_id")["capacity"].to_dict()
supply = node_params.set_index("node_id")["supply"].to_dict()
demand = node_params.set_index("node_id")["demand"].to_dict()

cost_source_dc = (
    arcs_df[arcs_df["arc_type"] == "source_dc"]
    .set_index(["from_node", "to_node"])["unit_transport_cost"]
    .to_dict()
)
cost_dc_market = (
    arcs_df[arcs_df["arc_type"] == "dc_market"]
    .set_index(["from_node", "to_node"])["unit_transport_cost"]
    .to_dict()
)

TOTAL_DEMAND = sum(demand[m] for m in markets)

UNMET_PENALTY_PER_UNIT = 150.0
DISRUPTION_PENALTY_PER_CAPACITY = 4.0
MITIGATION_EFFECTIVENESS = 0.50

hardening_cost = {j: 0.45 * fixed_cost[j] + 250 for j in dcs}
disruption_penalty = {j: DISRUPTION_PENALTY_PER_CAPACITY * capacity[j] for j in dcs}

print("Sources:", sources)
print("Candidate DCs:", dcs)
print("Markets:", markets)
print("Scenarios:", len(scenarios))
print("Total demand:", TOTAL_DEMAND)
print("Unmet penalty:", UNMET_PENALTY_PER_UNIT)
print("Hardening effectiveness:", MITIGATION_EFFECTIVENESS)


In [ ]:
# Cell 09: Import Gurobi for unmet-demand penalty sensitivity
try:
    import gurobipy as gp
    from gurobipy import GRB
    HAS_GUROBI = True
    print("Gurobi available.")
except Exception as e:
    HAS_GUROBI = False
    print("Gurobi unavailable. The unmet penalty sensitivity cell can only load existing results or be skipped.")
    print("Reason:", e)


In [ ]:
# Cell 10: Define Gurobi-based two-stage stochastic CVaR supply chain model
if not HAS_GUROBI:
    print("Gurobi is unavailable. Define functions only after installing gurobipy and activating a valid license.")

def solve_resilient_supply_chain_gurobi(
    objective="cost",
    cvar_limit=None,
    risk_dict=None,
    alpha=0.90,
    time_limit=300,
    mip_gap=1e-4,
    output_flag=0,
    fixed_open_dcs=None,
    fixed_hardened_dcs=None,
):
    if not HAS_GUROBI:
        raise RuntimeError("Gurobi is not available in this environment.")
    if risk_dict is None:
        risk_dict = p_is
    fixed_open_dcs = set(fixed_open_dcs) if fixed_open_dcs is not None else None
    fixed_hardened_dcs = set(fixed_hardened_dcs) if fixed_hardened_dcs is not None else None

    model = gp.Model("Scenario_Based_Resilient_SCND")
    model.Params.OutputFlag = output_flag
    model.Params.TimeLimit = time_limit
    model.Params.MIPGap = mip_gap

    y = model.addVars(dcs, vtype=GRB.BINARY, name="open_dc")
    r = model.addVars(dcs, vtype=GRB.BINARY, name="harden_dc")
    x = model.addVars(scenarios, sources, dcs, lb=0.0, vtype=GRB.CONTINUOUS, name="flow_source_dc")
    z = model.addVars(scenarios, dcs, markets, lb=0.0, vtype=GRB.CONTINUOUS, name="flow_dc_market")
    unmet = model.addVars(scenarios, markets, lb=0.0, vtype=GRB.CONTINUOUS, name="unmet_demand")
    eta = model.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="eta_cvar")
    excess = model.addVars(scenarios, lb=0.0, vtype=GRB.CONTINUOUS, name="cvar_excess")

    if fixed_open_dcs is not None:
        for j in dcs:
            model.addConstr(y[j] == (1 if j in fixed_open_dcs else 0), name=f"fix_open_{j}")
    if fixed_hardened_dcs is not None:
        for j in dcs:
            model.addConstr(r[j] == (1 if j in fixed_hardened_dcs else 0), name=f"fix_harden_{j}")

    for j in dcs:
        model.addConstr(r[j] <= y[j], name=f"harden_only_if_open_{j}")
    model.addConstr(gp.quicksum(y[j] for j in dcs) >= 2, name="minimum_two_dcs")

    for s in scenarios:
        for i in sources:
            p_si = float(risk_dict.get((s, i), 0.0))
            model.addConstr(
                gp.quicksum(x[s, i, j] for j in dcs) <= supply[i] * (1.0 - p_si),
                name=f"source_capacity_{s}_{i}",
            )
        for j in dcs:
            p_sj = float(risk_dict.get((s, j), 0.0))
            effective_capacity = capacity[j] * ((1.0 - p_sj) * y[j] + MITIGATION_EFFECTIVENESS * p_sj * r[j])
            model.addConstr(
                gp.quicksum(z[s, j, m] for m in markets) <= effective_capacity,
                name=f"dc_capacity_{s}_{j}",
            )
            model.addConstr(
                gp.quicksum(z[s, j, m] for m in markets) <= gp.quicksum(x[s, i, j] for i in sources),
                name=f"flow_conservation_{s}_{j}",
            )
        for m in markets:
            model.addConstr(
                gp.quicksum(z[s, j, m] for j in dcs) + unmet[s, m] >= demand[m],
                name=f"demand_satisfaction_{s}_{m}",
            )

    scenario_unmet = {s: gp.quicksum(unmet[s, m] for m in markets) for s in scenarios}
    for s in scenarios:
        model.addConstr(excess[s] >= scenario_unmet[s] - eta, name=f"cvar_excess_{s}")

    cvar_expr = eta + (1.0 / (1.0 - alpha)) * gp.quicksum(scenario_weights[s] * excess[s] for s in scenarios)
    if cvar_limit is not None:
        model.addConstr(cvar_expr <= cvar_limit, name="epsilon_cvar_limit")

    fixed_cost_expr = gp.quicksum(fixed_cost[j] * y[j] for j in dcs)
    hardening_cost_expr = gp.quicksum(hardening_cost[j] * r[j] for j in dcs)
    transport_expr = gp.quicksum(
        scenario_weights[s] * cost_source_dc[(i, j)] * x[s, i, j]
        for s in scenarios for i in sources for j in dcs
    ) + gp.quicksum(
        scenario_weights[s] * cost_dc_market[(j, m)] * z[s, j, m]
        for s in scenarios for j in dcs for m in markets
    )
    unmet_penalty_expr = gp.quicksum(
        scenario_weights[s] * UNMET_PENALTY_PER_UNIT * unmet[s, m]
        for s in scenarios for m in markets
    )
    disruption_penalty_expr = gp.quicksum(
        scenario_weights[s] * disruption_penalty[j]
        * (float(risk_dict.get((s, j), 0.0)) * y[j] - MITIGATION_EFFECTIVENESS * float(risk_dict.get((s, j), 0.0)) * r[j])
        for s in scenarios for j in dcs
    )
    total_cost_expr = fixed_cost_expr + hardening_cost_expr + transport_expr + unmet_penalty_expr + disruption_penalty_expr
    expected_unmet_expr = gp.quicksum(scenario_weights[s] * scenario_unmet[s] for s in scenarios)

    if objective == "cost":
        model.setObjective(total_cost_expr, GRB.MINIMIZE)
    elif objective == "cvar":
        model.setObjective(cvar_expr + 1e-6 * total_cost_expr, GRB.MINIMIZE)
    else:
        raise ValueError("objective must be 'cost' or 'cvar'.")

    model.optimize()
    status_map = {GRB.OPTIMAL: "Optimal", GRB.TIME_LIMIT: "TimeLimit", GRB.INFEASIBLE: "Infeasible"}
    status = status_map.get(model.status, f"Status_{model.status}")
    if model.SolCount == 0:
        return {"status": status, "model": model}

    return {
        "status": status,
        "objective": objective,
        "total_cost": float(total_cost_expr.getValue()),
        "fixed_cost": float(fixed_cost_expr.getValue()),
        "hardening_cost": float(hardening_cost_expr.getValue()),
        "expected_transport_cost": float(transport_expr.getValue()),
        "expected_unmet_penalty": float(unmet_penalty_expr.getValue()),
        "expected_disruption_penalty": float(disruption_penalty_expr.getValue()),
        "expected_unmet_demand": float(expected_unmet_expr.getValue()),
        "cvar_unmet_demand": float(cvar_expr.getValue()),
        "eta_cvar": float(eta.X),
        "open_dcs": [j for j in dcs if y[j].X > 0.5],
        "hardened_dcs": [j for j in dcs if r[j].X > 0.5],
        "model": model,
        "y_vars": y,
        "r_vars": r,
        "x_vars": x,
        "z_vars": z,
        "unmet_vars": unmet,
        "excess_vars": excess,
    }

print("Optimization model function defined.")


In [ ]:
# Cell 11: Define Pareto-frontier generation for any risk-input method
def solution_to_row(solution, method_name, point_id, alpha=0.90, cvar_limit=None):
    realized = extract_realized_solution_metrics(solution, scenarios, markets, scenario_weights, alpha=alpha)
    return {
        "risk_method": method_name,
        "point_id": point_id,
        "status": solution.get("status", None),
        "epsilon_cvar_limit": cvar_limit,
        "total_cost": solution.get("total_cost", np.nan),
        "fixed_cost": solution.get("fixed_cost", np.nan),
        "hardening_cost": solution.get("hardening_cost", np.nan),
        "expected_transport_cost": solution.get("expected_transport_cost", np.nan),
        "expected_unmet_penalty": solution.get("expected_unmet_penalty", np.nan),
        "expected_disruption_penalty": solution.get("expected_disruption_penalty", np.nan),
        "model_reported_cvar": solution.get("cvar_unmet_demand", np.nan),
        "realized_expected_unmet_demand": realized["realized_expected_unmet_demand"],
        "realized_cvar_unmet_demand": realized["realized_cvar_unmet_demand"],
        "realized_eta_cvar": realized["realized_eta_cvar"],
        "open_dcs": design_to_string(solution.get("open_dcs", [])),
        "hardened_dcs": design_to_string(solution.get("hardened_dcs", [])),
    }

def generate_pareto_for_risk_method(method_name, risk_dict, alpha=0.90, num_points=15, time_limit=300, mip_gap=1e-4):
    print(f"Generating Pareto frontier for {method_name}...")
    min_cost_solution = solve_resilient_supply_chain_gurobi(
        objective="cost", risk_dict=risk_dict, alpha=alpha, time_limit=time_limit, mip_gap=mip_gap, output_flag=0
    )
    min_cvar_solution = solve_resilient_supply_chain_gurobi(
        objective="cvar", risk_dict=risk_dict, alpha=alpha, time_limit=time_limit, mip_gap=mip_gap, output_flag=0
    )
    min_cost_realized = extract_realized_solution_metrics(min_cost_solution, scenarios, markets, scenario_weights, alpha=alpha)
    min_cvar_realized = extract_realized_solution_metrics(min_cvar_solution, scenarios, markets, scenario_weights, alpha=alpha)
    cvar_high = min_cost_realized["realized_cvar_unmet_demand"]
    cvar_low = min_cvar_realized["realized_cvar_unmet_demand"]
    if np.isnan(cvar_high) or np.isnan(cvar_low):
        raise RuntimeError(f"Failed to compute anchor CVaR values for {method_name}.")
    if abs(cvar_high - cvar_low) < 1e-6:
        eps_grid = [cvar_high]
    else:
        eps_grid = np.linspace(cvar_high, cvar_low, num_points)
    rows = []
    for point_id, eps in enumerate(eps_grid, start=1):
        sol = solve_resilient_supply_chain_gurobi(
            objective="cost", cvar_limit=float(eps) + 1e-6, risk_dict=risk_dict,
            alpha=alpha, time_limit=time_limit, mip_gap=mip_gap, output_flag=0
        )
        if sol.get("status") in {"Optimal", "TimeLimit"} and "unmet_vars" in sol:
            rows.append(solution_to_row(sol, method_name, point_id, alpha=alpha, cvar_limit=float(eps)))
        else:
            rows.append({"risk_method": method_name, "point_id": point_id, "status": sol.get("status"), "epsilon_cvar_limit": float(eps)})
    raw_df = pd.DataFrame(rows)
    nd_df = extract_nondominated(raw_df, cost_col="total_cost", cvar_col="realized_cvar_unmet_demand")
    return raw_df, nd_df


In [ ]:
# Cell 12: Sensitivity analysis on unmet demand penalty lambda

RUN_UNMET_PENALTY_SENSITIVITY = HAS_GUROBI
REUSE_EXISTING_UNMET_RESULTS = True

raw_file = find_file(
    "supp_sensitivity_unmet_penalty_raw_pareto.csv",
    patterns=["*unmet_penalty*raw*.csv"],
    required=False,
)

nd_file = find_file(
    "supp_sensitivity_unmet_penalty_nondominated_pareto.csv",
    patterns=["*unmet_penalty*nondominated*.csv"],
    required=False,
)

if REUSE_EXISTING_UNMET_RESULTS and raw_file is not None and nd_file is not None:
    unmet_raw = pd.read_csv(raw_file)
    unmet_nd = pd.read_csv(nd_file)
    print("Loaded existing unmet-penalty sensitivity results.")

elif RUN_UNMET_PENALTY_SENSITIVITY:
    UNMET_PENALTY_VALUES = [75.0, 150.0, 300.0]

    original_penalty = UNMET_PENALTY_PER_UNIT
    raw_list = []
    nd_list = []

    for penalty_value in UNMET_PENALTY_VALUES:
        print("Running unmet penalty sensitivity, lambda =", penalty_value)

        UNMET_PENALTY_PER_UNIT = penalty_value

        raw_df, nd_df = generate_pareto_for_risk_method(
            method_name=f"ML Dynamic_unmet_penalty_{penalty_value}",
            risk_dict=p_is,
            alpha=0.90,
            num_points=10,
            time_limit=300,
            mip_gap=1e-4,
        )

        raw_df["unmet_penalty_per_unit"] = penalty_value
        nd_df["unmet_penalty_per_unit"] = penalty_value

        raw_list.append(raw_df)
        nd_list.append(nd_df)

    UNMET_PENALTY_PER_UNIT = original_penalty

    unmet_raw = pd.concat(raw_list, ignore_index=True)
    unmet_nd = pd.concat(nd_list, ignore_index=True)

    save_csv(
        unmet_raw,
        "supp_sensitivity_unmet_penalty_raw_pareto.csv",
        also_root=True,
    )

    save_csv(
        unmet_nd,
        "supp_sensitivity_unmet_penalty_nondominated_pareto.csv",
        also_root=True,
    )

else:
    raise RuntimeError(
        "No existing unmet penalty sensitivity results found, and Gurobi is unavailable."
    )


# ---------------------------------------------------------------------
# Corrected summary:
# summarize unique nondominated design patterns, not raw epsilon-grid rows.
# ---------------------------------------------------------------------

required_cols = [
    "unmet_penalty_per_unit",
    "total_cost",
    "realized_expected_unmet_demand",
    "realized_cvar_unmet_demand",
    "open_dcs",
    "hardened_dcs",
]

missing_cols = [col for col in required_cols if col not in unmet_nd.columns]
if missing_cols:
    raise ValueError(
        f"Missing required columns in unmet-penalty nondominated Pareto file: {missing_cols}"
    )

unique_design_blocks = []

for penalty_value, group in unmet_nd.groupby("unmet_penalty_per_unit"):
    group = group.copy()

    group["open_dcs"] = group["open_dcs"].fillna("")
    group["hardened_dcs"] = group["hardened_dcs"].fillna("")

    # Round only for duplicate detection. Reported values remain unrounded.
    group["total_cost_round"] = group["total_cost"].round(6)
    group["expected_unmet_round"] = group["realized_expected_unmet_demand"].round(6)
    group["cvar_round"] = group["realized_cvar_unmet_demand"].round(6)

    # Step 1: remove exact or near-exact duplicates generated by repeated epsilon values.
    unique_group = (
        group.sort_values(
            by=[
                "total_cost",
                "realized_cvar_unmet_demand",
                "realized_expected_unmet_demand",
            ],
            ascending=[True, True, True],
        )
        .drop_duplicates(
            subset=[
                "open_dcs",
                "hardened_dcs",
                "total_cost_round",
                "expected_unmet_round",
                "cvar_round",
            ],
            keep="first",
        )
        .copy()
    )

    # Step 2: if the same first-stage design appears repeatedly at the same realized CVaR,
    # keep the lowest-cost realization.
    unique_group = (
        unique_group.sort_values(
            by=["total_cost", "realized_cvar_unmet_demand"],
            ascending=[True, True],
        )
        .drop_duplicates(
            subset=["open_dcs", "hardened_dcs", "cvar_round"],
            keep="first",
        )
        .copy()
    )

    unique_group["unique_design_id_within_penalty"] = range(1, len(unique_group) + 1)
    unique_design_blocks.append(unique_group)

unmet_unique_designs = pd.concat(unique_design_blocks, ignore_index=True)

unmet_unique_designs = unmet_unique_designs.drop(
    columns=[
        "total_cost_round",
        "expected_unmet_round",
        "cvar_round",
    ],
    errors="ignore",
)

unmet_summary = (
    unmet_unique_designs
    .groupby("unmet_penalty_per_unit")
    .agg(
        unique_nondominated_designs=("unique_design_id_within_penalty", "count"),
        min_cost=("total_cost", "min"),
        max_cost=("total_cost", "max"),
        min_cvar=("realized_cvar_unmet_demand", "min"),
        max_cvar=("realized_cvar_unmet_demand", "max"),
        min_expected_unmet=("realized_expected_unmet_demand", "min"),
        max_expected_unmet=("realized_expected_unmet_demand", "max"),
    )
    .reset_index()
    .sort_values("unmet_penalty_per_unit")
)

save_csv(
    unmet_unique_designs,
    "supp_sensitivity_unmet_penalty_nondominated_unique_designs.csv",
    also_root=True,
)

save_csv(
    unmet_summary,
    "supp_sensitivity_unmet_penalty_summary.csv",
    also_root=True,
)

design_patterns = (
    unmet_unique_designs
    .groupby(["unmet_penalty_per_unit", "open_dcs", "hardened_dcs"], dropna=False)
    .agg(
        count=("total_cost", "count"),
        min_cost=("total_cost", "min"),
        max_cost=("total_cost", "max"),
        min_cvar=("realized_cvar_unmet_demand", "min"),
        max_cvar=("realized_cvar_unmet_demand", "max"),
        min_expected_unmet=("realized_expected_unmet_demand", "min"),
        max_expected_unmet=("realized_expected_unmet_demand", "max"),
    )
    .reset_index()
    .sort_values(
        by=["unmet_penalty_per_unit", "min_cost", "min_cvar"],
        ascending=[True, True, True],
    )
)

save_csv(
    design_patterns,
    "supp_sensitivity_unmet_penalty_design_patterns.csv",
    also_root=True,
)

print("Corrected unmet-penalty sensitivity summary based on unique design patterns:")
display(unmet_summary)

print("Unique nondominated design patterns:")
display(
    unmet_unique_designs[
        [
            "unmet_penalty_per_unit",
            "unique_design_id_within_penalty",
            "total_cost",
            "realized_expected_unmet_demand",
            "realized_cvar_unmet_demand",
            "open_dcs",
            "hardened_dcs",
        ]
    ]
)


# ---------------------------------------------------------------------
# Plot unique nondominated designs.
# ---------------------------------------------------------------------

plt.figure(figsize=(8, 6))

for penalty_value in sorted(unmet_unique_designs["unmet_penalty_per_unit"].unique()):
    temp = (
        unmet_unique_designs[
            unmet_unique_designs["unmet_penalty_per_unit"] == penalty_value
        ]
        .sort_values(by="realized_cvar_unmet_demand")
        .copy()
    )

    plt.plot(
        temp["realized_cvar_unmet_demand"],
        temp["total_cost"],
        marker="o",
        label=f"$\\lambda$ = {penalty_value:g}",
    )

plt.xlabel("Realized CVaR of Unmet Demand")
plt.ylabel("Total Expected Cost")
plt.title("Sensitivity to Unmet Demand Penalty")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

save_figure("supp_sensitivity_unmet_penalty_pareto.png")
plt.show()

In [ ]:
# Cell 14: Final supplementary output manifest and expected files

expected_outputs = [
    "supp_temporal_holdout_model_results_2000_2020_to_2021_2025.csv",
    "supp_threshold_positive_ratio_summary.csv",
    "supp_threshold_rf_groupcv_available_thresholds.csv",
    "supp_ml_probability_vs_rule_examples.csv",
    "supp_ml_probability_distribution_by_rule_label.csv",
    "supp_scenario_reduction_diagnostics.csv",
    "supp_selected_scenarios_by_reduction_method.csv",
    "supp_scenario_reduction_diagnostic_plot.png",
    "supp_sensitivity_unmet_penalty_raw_pareto.csv",
    "supp_sensitivity_unmet_penalty_nondominated_pareto.csv",
    "supp_sensitivity_unmet_penalty_summary.csv",
    "supp_sensitivity_unmet_penalty_design_patterns.csv",
    "supp_sensitivity_unmet_penalty_pareto.png",
]

manifest_rows = []
for name in expected_outputs:
    root_path = BASE_DIR / name
    out_path = OUTPUT_DIR / name
    manifest_rows.append({
        "file": name,
        "exists_in_root": root_path.exists(),
        "exists_in_outputs_03": out_path.exists(),
        "root_path": str(root_path) if root_path.exists() else "",
        "outputs_03_path": str(out_path) if out_path.exists() else "",
    })
manifest_df = pd.DataFrame(manifest_rows)
save_csv(manifest_df, "supplementary_output_manifest.csv", also_root=True)
display_if_available(manifest_df, max_rows=50)

print("Files currently in outputs_03_supplementary:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p.name)
